# OpenClaw + Ernie 5.0 (AI Studio 云端部署)
## [参考项目：在OpenClaw中使用Ernie5.0](https://aistudio.baidu.com/projectdetail/10013178)
## 项目目标
本项目旨在指导用户在 **百度 AI Studio** 容器环境下，部署 **OpenClaw** 核心网关。通过接入 **Ernie 5.0 (文心一言)** 大模型接口，并利用 **ngrok** 实现内网穿透，打造一个可供公网远程设备（如各类边缘节点或 IoT 硬件）接入协同的云端 AI 控制中枢。

## 测试环境
- **运行平台**: 百度 AI Studio (Ubuntu 容器环境)
- **核心组件**:
  - Ernie 5.0 Thinking Preview (对话大模型)
  - Node.js 22 (通过 NVM 环境隔离安装)
  - ngrok (公网隧道穿透)
  - OpenClaw Latest (Agent 编排与设备管理)

## 部署指引

部署过程需要开启三个独立的终端 (Terminal) 并行操作：

### 终端 1：环境配置与网关启动
在第一个终端中，我们将配置 Node.js 环境，安装 OpenClaw，并连接文心一言大模型。

```bash
# 1. 安装 NVM 并配置 Node.js 22 环境
curl -o- https://raw.githubusercontent.com/nvm-sh/nvm/v0.39.7/install.sh | bash 或者 bash install_nvm.sh
source ~/.bashrc
nvm install 22
nvm use 22
node --version

# 2. 安装并初始化 OpenClaw
npm install -g openclaw@latest
openclaw onboard --install-daemon
openclaw setup

# 3. 基础与安全配置
openclaw config set gateway.mode local
# 添加允许的域名与信任代理，保障穿透后的 UI 访问
openclaw config set gateway.controlUi.allowedOrigins '["你的域名"]'
openclaw config set gateway.trustedProxies '["127.0.0.1"]'
# 禁用认证（开发/测试环境下可选，简化流程）
openclaw config set gateway.auth.mode "none"

# 4. 配置 Ernie 5.0 (文心一言) 接入点
openclaw config set models.providers.ernie-profile '{
  "baseUrl": "https://aistudio.baidu.com/llm/lmapi/v3",
  "apiKey": "你的_API_KEY",
  "api": "openai-completions",
  "models": [
    { "id": "ernie-5.0-thinking-preview", "name": "Ernie 5.0 Thinking" }
  ]
}'
openclaw config set agents.defaults.model.primary "ernie-profile/ernie-5.0-thinking-preview"

# 5. 启动网关服务 (监听 18789 端口)
openclaw gateway --port 18789 --verbose


```

![](https://ai-studio-static-online.cdn.bcebos.com/86105d986c8340ffbb681abb9e4d2713a97a01758dea49ae85566e3f18053f02)
![](https://ai-studio-static-online.cdn.bcebos.com/63bb2f7e619d488682b17943cbd81a5956b40556f5db4342954455004e31dbf0)
![](https://ai-studio-static-online.cdn.bcebos.com/867957be8b444cdf86a693249ff73662530e201359c540d7b72154880e90594e)
![](https://ai-studio-static-online.cdn.bcebos.com/725f90cdc0674ba6977a57b427e3577c5bc533aa8fc04add9325850dcae32bd7)



### 终端 2：内网穿透 (ngrok) 不一定非要用ngrok，frp也可以
保持终端 1 运行，打开第二个终端。将 AI Studio 内部的 18789 端口映射到公网，以便外部设备访问控制面板和网关。
#### 解压ngrok
tar -xzf ngrok-v3-stable-linux-amd64.tgz
#### 配置你的 ngrok 授权码
./ngrok config add-authtoken 你的_NGROK_TOKEN

#### 启动 HTTP 隧道，绑定固定域名及本地端口
./ngrok http --domain=你的域名 18789

![](https://ai-studio-static-online.cdn.bcebos.com/2c5d1f178e7e4cb582e26e5eee9b875981fb044adcc04641bfb60d995c367db4)


### 终端 3：设备接入与授权
#### 列出当前所有的设备服务申请列表
openclaw devices list

#### 批准特定设备 UUID 接入网关
openclaw devices approve request下设备UUID
![](https://ai-studio-static-online.cdn.bcebos.com/138a92f3c3284c438f683a37b993983aec399d8ddff740da99c80eb914b2b775)


![](https://ai-studio-static-online.cdn.bcebos.com/bd9acfca2770477b8ccabb0290d42ede24b124cb15b64010b2c6aceb06f493db)
![](https://ai-studio-static-online.cdn.bcebos.com/7697fd33a0954f219986ec13bd4f1d98f3dcf9433dc1422781df9407d135d25c)
![](https://ai-studio-static-online.cdn.bcebos.com/e930cf56820546d193438619d7f0c03cca92a4e3a9e64d75ba7c180e42bd7d24)
